<div style="border-top:5px solid #175569;padding:18px 0 8px">
<div style="font-size:12px;letter-spacing:2px;color:#536774">PROFESSOR NANO LABS · CRYSTALS &amp; ENERGY BANDS</div>
<h1 style="margin:10px 0 6px">From atoms to a band</h1>
<div style="font-size:21px;color:#175569">One dimensional lattice model &amp; density of states</div>
<p style="color:#536774">Professor Tony Low · University of Minnesota</p>
</div>

**The central question:** How can one localized state per atom produce an energy band—and why are the states not distributed uniformly in energy?

**Try it live in your browser:** [From Atoms to a Band lab](https://tonylowgroup.github.io/Professor-Nano-Labs/labs/atoms-to-a-band/)  
**Watch first:** [How energy bands are formed? Key quantum concepts simplified](https://www.youtube.com/watch?v=Hm-v4Eby0hk)  
**Watch later:** [Deriving the density-of-states for any dispersion and dimensionality](https://www.youtube.com/watch?v=vNBttmWca_U) · [Linear combination of atomic orbitals (tight-binding method)](https://www.youtube.com/watch?v=5vOvjeTRcqM)

This notebook covers the one-dimensional lattice model and its density of states. It stops before wavefunctions and Bloch's theorem, and it does not include the overlap between neighbouring orbitals.

We will build the Hamiltonian, add atoms, vary the coupling, and count the resulting energy levels. The browser experiments and GIFs are **parameter sweeps, not quantum time evolution**.

## Start here

Run **Kernel → Restart Kernel and Run All Cells** (or **Run All**). This copy is saved without outputs to keep the download small; running it builds the browser experiments and three GIFs, and saves them into `atoms_to_a_band_outputs` in your working directory: one standalone HTML file per experiment (`atoms_to_a_band_band.html`, `..._spacing.html`, `..._dos.html`) plus the GIFs.

The browser controls provide immediate updates without `ipywidgets`, Plotly, SciPy, or WebGL. The independent Python calculations use only **NumPy, Matplotlib, Pillow, and IPython**. All browser code and GIF images are embedded; no internet is needed after the Python packages are installed.

If a viewer blocks the embedded browser controls, trust and rerun your local notebook, open the exported HTML files in a regular browser, or use the [online lab](https://tonylowgroup.github.io/Professor-Nano-Labs/labs/atoms-to-a-band/). The GIF outputs are a further fallback. Browser sliders do **not** modify the Python parameter cell below; edit that cell and rerun to change the Python examples and exports.

In [ ]:
from __future__ import annotations
import importlib.util
import sys

_required = {"numpy": "numpy", "matplotlib": "matplotlib", "Pillow": "PIL"}
_missing = [package for package, module in _required.items()
            if importlib.util.find_spec(module) is None]
if _missing:
    raise ImportError(
        "Missing packages: " + ", ".join(_missing) +
        ". Run `%pip install numpy matplotlib pillow` in a new notebook cell, "
        "then restart the kernel and Run All. Installation requires internet access."
    )

import base64
import html
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image as PILImage
from IPython.display import HTML, Image, display

%matplotlib inline
plt.ioff()
print(f"Python {sys.version.split()[0]} | NumPy {np.__version__}")
print("Ready. No Plotly, SciPy, widget extension, or web connection is used by the lab.")

## 0. Model choices and editable parameters

We use **one orbital per site**, an **open chain**, a uniform on-site energy $\varepsilon$, and a **real negative** nearest-neighbor matrix element $t$. Thus the $n=1$ state is the lowest-energy state. We count **one spin channel** and use the DOS length convention **$L=Na$**.

Energies are in **eV** and lengths in **Å**. The defaults $a=5$ Å and $t=-0.4$ eV give an illustrative band about 1.6 eV wide. The reference hopping and $\kappa$ below are **illustrative model inputs**, not values fitted to a particular material.

The numerical matrix treats the local basis as orthonormal, as in the simplest tight-binding treatment. No overlap correction is introduced.

In [ ]:
# EDIT THIS CELL, then Run All, to change the Python calculations and exports.
SETTINGS = {
    "N": 32,                    # one orbital at each of N sites; browser: 1–256
    "epsilon_eV": 0.0,          # epsilon: shifting it shifts every level equally
    "t_eV": -0.40,              # t < 0; the browser also has an isolated t=0 limit
    "a_A": 5.0,                 # lattice spacing used in L = N*a
    "a_ref_A": 5.0,             # reference spacing for the spacing experiment
    "t_ref_eV": -0.40,          # hopping at a_ref
    "kappa_invA": 1.0,          # orbital-tail decay scale in the approximate t(a)
    "eta_over_abs_t": 0.08,     # Gaussian sigma / |t|; display resolution only
}
MAKE_GIFS = True                # False skips GIF regeneration during quick edits
OUTPUT_DIR = Path.cwd() / "atoms_to_a_band_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Keep the exported browser controls and their initial values consistent.
assert isinstance(SETTINGS["N"], int) and not isinstance(SETTINGS["N"], bool)
assert 1 <= SETTINGS["N"] <= 256, "Use N = 1, ..., 256 for this browser lab."
assert -1.2 <= SETTINGS["t_eV"] <= -0.05, "Use negative, nonzero t here; t=0 is available in Experiment 1."
assert -1 <= SETTINGS["epsilon_eV"] <= 1
assert 3 <= SETTINGS["a_A"] <= 10
assert SETTINGS["a_ref_A"] > 0 and SETTINGS["t_ref_eV"] < 0
assert 0.6 <= SETTINGS["kappa_invA"] <= 1.8
assert 0.02 <= SETTINGS["eta_over_abs_t"] <= 0.30
assert all(np.isfinite(v) for v in SETTINGS.values())
print(f"N = {SETTINGS['N']}, epsilon = {SETTINGS['epsilon_eV']:.2f} eV, "
      f"t = {SETTINGS['t_eV']:.2f} eV, a = {SETTINGS['a_A']:.2f} Å")

## 1. The one dimensional lattice Hamiltonian

The two matrix elements are

$$\varepsilon=\langle j|H|j\rangle,\qquad t=\langle j|H|j+1\rangle.$$

For a real $t$ and open ends,

$$H=\begin{pmatrix}
\varepsilon&t&0&\cdots&0\\
t&\varepsilon&t&\ddots&\vdots\\
0&t&\varepsilon&\ddots&0\\
\vdots&\ddots&\ddots&\ddots&t\\
0&\cdots&0&t&\varepsilon
\end{pmatrix},\qquad H|c_n\rangle=E_n|c_n\rangle.$$

The exact finite-chain spectrum is

$$\boxed{E_n=\varepsilon+2t\cos\left(\frac{n\pi}{N+1}\right)},\qquad n=1,\ldots,N.$$

The next cell implements both routes: **construct and diagonalize $H$ numerically**, and **evaluate the exact formula**. The browser uses the exact formula for fast updates; it is not replacing the matrix with a different physical model.

In [ ]:
def _check_N(N):
    """Return a valid number of sites (one orbital per site)."""
    if isinstance(N, (bool, np.bool_)) or not isinstance(N, (int, np.integer)) or N < 1:
        raise ValueError("N must be an integer greater than or equal to 1.")
    return int(N)


def _check_real(value, name):
    if not np.isscalar(value) or not np.isreal(value) or not np.isfinite(value):
        raise ValueError(f"{name} must be a finite real number.")
    return float(value)


def make_hamiltonian(N, epsilon_eV=0.0, t_eV=-0.4):
    """Open, uniform, nearest-neighbor chain; real t; energies in eV."""
    N = _check_N(N)
    epsilon_eV = _check_real(epsilon_eV, "epsilon_eV")
    t_eV = _check_real(t_eV, "t_eV")
    # Put epsilon on the diagonal and t on both
    # nearest-neighbor diagonals. These three lines construct the model H.
    H = epsilon_eV * np.eye(N)
    H += np.diag(np.full(N - 1, t_eV), k=1)
    H += np.diag(np.full(N - 1, t_eV), k=-1)
    # No extra end-to-end bond: this is an OPEN chain, not a periodic ring.
    return H


def exact_levels(N, epsilon_eV=0.0, t_eV=-0.4):
    """The exact cosine spectrum, sorted by energy."""
    N = _check_N(N)
    epsilon_eV = _check_real(epsilon_eV, "epsilon_eV")
    t_eV = _check_real(t_eV, "t_eV")
    n = np.arange(1, N + 1)
    # THIS IS THE EXACT SPECTRUM: E_n = epsilon + 2t cos[n*pi/(N+1)].
    levels = epsilon_eV + 2.0 * t_eV * np.cos(n * np.pi / (N + 1))
    return np.sort(levels)


def hopping_from_spacing(a_A, t_ref_eV=-0.4, a_ref_A=5.0, kappa_invA=1.0):
    """Normalized version of the approximate a*exp(-kappa*a) hopping.

    t_ref and kappa are illustrative independent inputs. This is NOT an
    exact solution of the continuum delta-well lattice or a material fit.
    """
    a_A = np.asarray(a_A, dtype=float)
    if np.any(~np.isfinite(a_A)) or np.any(a_A <= 0):
        raise ValueError("All lattice spacings must be positive and finite.")
    t_ref_eV = _check_real(t_ref_eV, "t_ref_eV")
    a_ref_A = _check_real(a_ref_A, "a_ref_A")
    kappa_invA = _check_real(kappa_invA, "kappa_invA")
    if a_ref_A <= 0 or kappa_invA <= 0:
        raise ValueError("a_ref_A and kappa_invA must be positive.")
    return t_ref_eV * (a_A / a_ref_A) * np.exp(-kappa_invA * (a_A - a_ref_A))


def cumulative_count(E_eV, levels_eV):
    """Omega_N(E): exact number of finite-chain states at or below E."""
    levels_eV = np.asarray(levels_eV, dtype=float)
    if levels_eV.ndim != 1 or levels_eV.size == 0 or not np.all(np.isfinite(levels_eV)):
        raise ValueError("levels_eV must be a nonempty, finite, one-dimensional array.")
    return np.searchsorted(np.sort(levels_eV), E_eV, side="right")


def gaussian_dos(E_eV, levels_eV, a_A, sigma_eV):
    """Numerical D_sigma(E), per length, in states/(eV Angstrom).

    L = N*a exactly, and every Gaussian has unit area before division by L.
    sigma is a DISPLAY resolution, not a temperature or a fitted lifetime.
    """
    E_eV = np.asarray(E_eV, dtype=float)
    levels_eV = np.asarray(levels_eV, dtype=float)
    a_A = _check_real(a_A, "a_A")
    sigma_eV = _check_real(sigma_eV, "sigma_eV")
    if E_eV.ndim != 1 or levels_eV.ndim != 1 or levels_eV.size == 0:
        raise ValueError("E_eV and levels_eV must be 1D; the level array cannot be empty.")
    if not np.all(np.isfinite(E_eV)) or not np.all(np.isfinite(levels_eV)):
        raise ValueError("Energies must be finite.")
    if a_A <= 0 or sigma_eV <= 0:
        raise ValueError("a_A and sigma_eV must be positive.")
    L_A = levels_eV.size * a_A
    z = (E_eV[:, None] - levels_eV[None, :]) / sigma_eV
    # Sum delta(E-E_n)/L, with each delta displayed
    # as a NORMALIZED Gaussian so the total number of states is preserved.
    return np.exp(-0.5 * z**2).sum(axis=1) / (L_A * sigma_eV * np.sqrt(2.0 * np.pi))


def bulk_dos(E_eV, epsilon_eV=0.0, t_eV=-0.4, a_A=5.0):
    """Ideal, unbroadened large-N DOS, per length; one spin channel.

    Zero outside the band; +infinity exactly at an edge. Plotting routines
    omit infinities rather than substituting a finite physical edge value.
    """
    E = np.asarray(E_eV, dtype=float)
    epsilon_eV = _check_real(epsilon_eV, "epsilon_eV")
    t_eV = _check_real(t_eV, "t_eV")
    a_A = _check_real(a_A, "a_A")
    if t_eV == 0 or a_A <= 0:
        raise ValueError("Bulk formula requires nonzero t and positive a. At t=0 use delta(E-epsilon)/a.")
    rad = (2.0 * abs(t_eV))**2 - (E - epsilon_eV)**2
    D = np.zeros_like(E, dtype=float)
    inside = rad > 0
    D[inside] = 1.0 / (np.pi * a_A * np.sqrt(rad[inside]))
    edges = (E == epsilon_eV - 2.0 * abs(t_eV)) | (E == epsilon_eV + 2.0 * abs(t_eV))
    D[edges] = np.inf
    return D


def bulk_cdf(E_eV, epsilon_eV=0.0, t_eV=-0.4):
    """Fraction of bulk-band states at or below E: ranges from 0 to 1."""
    E = np.asarray(E_eV, dtype=float)
    if t_eV == 0:
        return (E >= epsilon_eV).astype(float)
    return np.arccos(np.clip((epsilon_eV - E) / (2.0 * abs(t_eV)), -1.0, 1.0)) / np.pi


def smoothed_bulk_dos(E_eV, epsilon_eV=0.0, t_eV=-0.4,
                      a_A=5.0, sigma_eV=0.032, n_q=2048):
    """Apply the SAME display broadening to the continuum reference.

    Midpoint quadrature in q avoids sampling the ideal DOS singularities:
        D_bulk,sigma(E) = (1/(pi*a)) integral_0^pi G_sigma(E-E(q)) dq.
    q is the continuum limit of n*pi/(N+1), not an added boundary condition.
    """
    n_q = _check_N(n_q)
    q = (np.arange(n_q) + 0.5) * np.pi / n_q
    continuum_energies = epsilon_eV + 2.0 * t_eV * np.cos(q)
    return gaussian_dos(np.asarray(E_eV), continuum_energies, a_A, sigma_eV)


def integrate_trapezoid(y, x):
    """Trapezoidal integral; works without NumPy-version-specific aliases."""
    y, x = np.asarray(y), np.asarray(x)
    return np.sum((y[1:] + y[:-1]) * np.diff(x) / 2.0)


# This separate numerical diagonalization does NOT use the cosine formula.
H_demo = make_hamiltonian(4, SETTINGS["epsilon_eV"], SETTINGS["t_eV"])
E_numeric = np.linalg.eigvalsh(H_demo)
E_formula = exact_levels(4, SETTINGS["epsilon_eV"], SETTINGS["t_eV"])
print("H for four sites (eV):")
print(np.array2string(H_demo, precision=3, suppress_small=True))
print("\nNumerical eigenvalues (eV):", np.round(E_numeric, 6))
print("Cosine formula       (eV):", np.round(E_formula, 6))
print(f"Maximum disagreement: {np.max(np.abs(E_numeric-E_formula)):.2e} eV")

### Browser and GIF helpers
The next two implementation cells handle browser drawing and image export. Their source is marked as folded in notebook frontends that support that setting. The physics functions above are the main code to study; these helpers need not be modified.

In [ ]:
LAB_TEMPLATE = '<!doctype html>\n<html lang="en">\n<head>\n<meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1">\n<title>From atoms to a band · Professor Nano Labs</title>\n<style>\n:root { color-scheme:light; --ink:#162d3d; --muted:#536774; --paper:#fff; --line:#d9e2e6; --soft:#f4f7f8; --accent:#175569; }\n*{box-sizing:border-box}body{margin:0;padding:18px;font:15px/1.5 -apple-system,BlinkMacSystemFont,"Segoe UI",Arial,sans-serif;color:var(--ink);background:var(--paper)}\nbutton,input,select{font:inherit}button,select{border:1px solid var(--line);border-radius:7px;background:white;color:var(--ink);padding:7px 11px}button{cursor:pointer}button:hover{background:var(--soft)}button:focus-visible,input:focus-visible,select:focus-visible{outline:3px solid #77a5b5;outline-offset:2px}button.primary{background:var(--accent);border-color:var(--accent);color:white}button.active{box-shadow:inset 0 0 0 1px var(--accent)}\nh2{margin:0 0 7px;font-size:24px;line-height:1.2;letter-spacing:-.4px}h3{font-size:15px;margin:0 0 5px}p{margin:6px 0 10px}.kicker{font-size:11px;font-weight:700;letter-spacing:1.7px;text-transform:uppercase;color:var(--accent);margin-bottom:8px}.sub{color:var(--muted);max-width:1000px}.equation{font:17px/1.6 Georgia,"Times New Roman",serif;padding:10px 13px;border-left:3px solid var(--accent);background:var(--soft);margin:13px 0}.small{font-size:12px;color:var(--muted)}\n.controls{display:grid;grid-template-columns:repeat(3,minmax(0,1fr));gap:13px 22px;padding:15px;background:var(--soft);border:1px solid var(--line);border-radius:11px;margin:14px 0}.control label{display:flex;justify-content:space-between;gap:8px;font-size:13px;font-weight:600;margin-bottom:4px}.control output{font-weight:700;font-variant-numeric:tabular-nums;white-space:nowrap}.control input[type=range]{width:100%;accent-color:var(--accent);cursor:pointer}.control select{width:100%;padding:4px 7px}.player{display:flex;align-items:center;flex-wrap:wrap;gap:7px;margin:12px 0}.player .small{margin-left:4px}.preset{display:flex;align-items:center;flex-wrap:wrap;gap:5px;margin:7px 0 12px}.preset button{font-size:12px;padding:4px 9px}.preset span{font-size:12px;margin-right:4px;color:var(--muted)}\n.metrics{display:grid;grid-template-columns:repeat(4,minmax(0,1fr));gap:10px;margin:13px 0}.metric{border:1px solid var(--line);border-radius:9px;padding:10px 12px;min-width:0}.metric .key{font-size:11px;text-transform:uppercase;letter-spacing:.65px;color:var(--muted)}.metric .value{font-size:22px;font-weight:700;margin-top:2px;line-height:1.3;overflow-wrap:anywhere;font-variant-numeric:tabular-nums}.metric .foot{font-size:11px;color:var(--muted)}\n.plotgrid{display:grid;grid-template-columns:repeat(2,minmax(0,1fr));gap:13px}.plot{border:1px solid var(--line);border-radius:11px;padding:11px;min-width:0;overflow:hidden}.plot.wide{grid-column:1/-1}.plot svg{display:block;width:100%;height:auto;overflow:visible}.plot .caption{font-size:12px;color:var(--muted);margin:4px 2px}.lattice{margin:10px 0;border-bottom:1px solid var(--line)}.lattice svg{display:block;width:100%;max-height:118px}.legend{display:flex;flex-wrap:wrap;gap:7px 17px;font-size:12px;margin:3px 0 7px}.legend span{display:inline-flex;align-items:center;gap:6px}.swatch{display:inline-block;width:23px;border-top:2px solid var(--s1)}.swatch.ideal{border-top-color:var(--s2);border-top-style:dashed}.swatch.smooth{border-top-color:var(--s3);border-top-style:dotted}.swatch.selected{width:10px;height:10px;border:0;border-radius:50%;background:var(--s2)}.note{background:var(--soft);border-radius:8px;padding:10px 12px;margin:13px 0;font-size:13px}.status{font-size:12px;color:var(--muted);margin:12px 0 1px}details{border-top:1px solid var(--line);margin-top:12px;padding-top:8px;font-size:13px}summary{cursor:pointer;font-weight:600}.axis{stroke:#7f929d;stroke-width:1}.gridline{stroke:#e5ebee;stroke-width:1}.tick{font-size:12px;fill:var(--muted)}.axislabel{font-size:14px;fill:var(--ink)}.series{fill:none;stroke:var(--s1);stroke-width:2.3;stroke-linejoin:round}.ideal-line{fill:none;stroke:var(--s2);stroke-width:1.8;stroke-dasharray:7 5}.smooth-line{fill:none;stroke:var(--s3);stroke-width:2;stroke-dasharray:2 4}.guide{stroke:var(--muted);stroke-width:1;stroke-dasharray:4 4;opacity:.6}.selected-band{fill:var(--s2);opacity:.12}.point{fill:var(--s1)}.selected-point{fill:var(--s2);stroke:white;stroke-width:.4}.mutedpoint{fill:var(--s1);opacity:.45}.fan{fill:none;stroke:var(--s1);stroke-width:.9;opacity:.65}.matrixzero{fill:#94a3ac}.matrixval{fill:var(--ink);font-weight:600}.matrixdiag{fill:var(--soft)}.matrixbond{fill:var(--s1);opacity:.12}\nbody[data-mode="dos"] .plot:not(.wide) h3{min-height:42px}\n@media(max-width:650px){body{padding:10px}.controls{grid-template-columns:repeat(2,minmax(0,1fr));gap:10px}.metrics{grid-template-columns:repeat(2,minmax(0,1fr))}.plotgrid{grid-template-columns:1fr}h2{font-size:21px}.plot.wide{grid-column:auto}}\n</style>\n</head>\n<body>\n<div class="kicker" id="kicker"></div><h2 id="title"></h2><p class="sub" id="intro"></p><div class="equation" id="formula"></div>\n<div class="controls" id="controls"></div><div id="presets"></div>\n<div class="player"><button type="button" class="primary" id="play">▶ Play</button><button type="button" id="reverse">↶ Reverse</button><button type="button" id="back">◀ Step</button><button type="button" id="forward">Step ▶</button><button type="button" id="reset">Reset</button><label class="small" for="speed">Speed</label><select id="speed" aria-label="Animation speed"><option value="0.5">0.5×</option><option value="1" selected>1×</option><option value="2">2×</option><option value="4">4×</option></select><span class="small" id="sweepnote"></span></div>\n<div id="metrics" class="metrics" aria-live="polite"></div><div id="lattice" class="lattice"></div><div class="plotgrid" id="plots"></div><div class="note" id="takeaway"></div><details><summary>What is being computed?</summary><div id="method"></div></details><div class="status" id="status"></div>\n<script>\n\'use strict\';\n// No external scripts, network access, Plotly, widgets, or WebGL are needed.\nconst CFG = __CONFIG__;\nconst MODE = __MODE__;\nconst $ = id => document.getElementById(id);\nconst esc = s => String(s).replace(/[&<>"\']/g, c => ({\'&\':\'&amp;\',\'<\':\'&lt;\',\'>\':\'&gt;\',\'"\':\'&quot;\',"\'":\'&#39;\'}[c]));\nconst initial = {N:CFG.N, mag:Math.abs(CFG.t_eV), eps:CFG.epsilon_eV, a:CFG.a_A,\n                 kappa:CFG.kappa_invA, eta:CFG.eta_over_abs_t, center:0,\n                 width:0.10, norm:\'length\', sweep:\'N\'};\nlet state = {...initial};\nlet playing=false, direction=1, raf=null, lastFrame=0, speed=1;\nlet smoothCache=null;\nconst Ns=[1,2,3,4,8,16,32,64,128,256];\nconst colors=CFG.plot_colors;\ndocument.body.dataset.mode=MODE;\ndocument.documentElement.style.setProperty(\'--s1\',colors[0]);\ndocument.documentElement.style.setProperty(\'--s2\',colors[1]);\ndocument.documentElement.style.setProperty(\'--s3\',colors[2]);\nfunction fmt(x,d=3){if(!Number.isFinite(x))return String(x);if(x===0)return \'0\';if(Math.abs(x)<0.001 || Math.abs(x)>=10000)return x.toExponential(2);return Number(x.toPrecision(d)).toString();}\nfunction fixed(x,d=2){return Number(x).toFixed(d);}\nfunction grid(a,b,n){return Array.from({length:n},(_,i)=>a+(b-a)*i/(n-1));}\n// Exact solution of the OPEN tridiagonal matrix.\nfunction levels(N,eps,t){return Array.from({length:N},(_,i)=>eps+2*t*Math.cos((i+1)*Math.PI/(N+1))).sort((a,b)=>a-b);}\nfunction hopping(a){return CFG.t_ref_eV*(a/CFG.a_ref_A)*Math.exp(-state.kappa*(a-CFG.a_ref_A));}\nfunction countAt(E,energies){let lo=0,hi=energies.length;while(lo<hi){const mid=(lo+hi)>>1;if(energies[mid]<=E)lo=mid+1;else hi=mid;}return lo;}\nfunction integral(y,x){let r=0;for(let i=1;i<x.length;i++)r+=(y[i]+y[i-1])*(x[i]-x[i-1])/2;return r;}\n// Unit-area Gaussian in reduced energy u=(E-epsilon)/|t|.\nfunction densityShape(us,rs,r){const denom=rs.length*r*Math.sqrt(2*Math.PI);return us.map(u=>{let s=0;for(const e of rs){const z=(u-e)/r;if(Math.abs(z)<9)s+=Math.exp(-.5*z*z);}return s/denom;});}\nfunction niceTicks(lo,hi,target=5){const raw=(hi-lo)/target;const scale=10**Math.floor(Math.log10(raw));const a=raw/scale;const step=(a<=1?1:a<=2?2:a<=2.5?2.5:a<=5?5:10)*scale;const ticks=[];for(let v=Math.ceil(lo/step)*step;v<=hi+step*1e-7;v+=step)ticks.push(Math.abs(v)<step*1e-8?0:v);return ticks;}\nfunction plot(id,xlim,ylim,xlabel,ylabel,wide=false){const W=wide?1040:550,H=wide?315:380;const M={l:77,r:20,t:17,b:58};const x=v=>M.l+(v-xlim[0])/(xlim[1]-xlim[0])*(W-M.l-M.r);const y=v=>H-M.b-(v-ylim[0])/(ylim[1]-ylim[0])*(H-M.t-M.b);const clip=id+\'-clip\';let body=`<defs><clipPath id="${clip}"><rect x="${M.l}" y="${M.t}" width="${W-M.l-M.r}" height="${H-M.t-M.b}"/></clipPath></defs>`;\nfor(const v of niceTicks(...xlim)){const xx=x(v);body+=`<line class="gridline" x1="${xx}" y1="${M.t}" x2="${xx}" y2="${H-M.b}"/><text class="tick" x="${xx}" y="${H-M.b+23}" text-anchor="middle">${fmt(v)}</text>`;}\nfor(const v of niceTicks(...ylim)){const yy=y(v);body+=`<line class="gridline" x1="${M.l}" y1="${yy}" x2="${W-M.r}" y2="${yy}"/><text class="tick" x="${M.l-10}" y="${yy+4}" text-anchor="end">${fmt(v)}</text>`;}\nbody+=`<line class="axis" x1="${M.l}" y1="${H-M.b}" x2="${W-M.r}" y2="${H-M.b}"/><line class="axis" x1="${M.l}" y1="${M.t}" x2="${M.l}" y2="${H-M.b}"/><text class="axislabel" x="${(M.l+W-M.r)/2}" y="${H-10}" text-anchor="middle">${esc(xlabel)}</text><text class="axislabel" transform="translate(19 ${(M.t+H-M.b)/2}) rotate(-90)" text-anchor="middle">${esc(ylabel)}</text>`;\nlet content=\'\';const api={x,y,xlim,ylim,clip,\n path(xs,ys,klass=\'series\'){let p=\'\',pen=false;for(let i=0;i<xs.length;i++){if(!Number.isFinite(xs[i])||!Number.isFinite(ys[i])){pen=false;continue;}p+=(pen?\'L\':\'M\')+x(xs[i]).toFixed(2)+\',\'+y(ys[i]).toFixed(2);pen=true;}content+=`<path class="${klass}" d="${p}"/>`;},\n dots(xs,ys,selected=null){for(let i=0;i<xs.length;i++)content+=`<circle class="${selected?(selected[i]?\'selected-point\':\'mutedpoint\'):\'point\'}" cx="${x(xs[i])}" cy="${y(ys[i])}" r="${xs.length>100?2.2:3.3}"/>`;},\n hline(v,klass=\'guide\'){content+=`<line class="${klass}" x1="${x(xlim[0])}" x2="${x(xlim[1])}" y1="${y(v)}" y2="${y(v)}"/>`;},\n vline(v,klass=\'guide\'){content+=`<line class="${klass}" x1="${x(v)}" x2="${x(v)}" y1="${y(ylim[0])}" y2="${y(ylim[1])}"/>`;},\n shadeY(a,b){content+=`<rect class="selected-band" x="${x(xlim[0])}" y="${y(b)}" width="${x(xlim[1])-x(xlim[0])}" height="${y(a)-y(b)}"/>`;},\n shadeX(a,b){content+=`<rect class="selected-band" x="${x(a)}" y="${y(ylim[1])}" width="${x(b)-x(a)}" height="${y(ylim[0])-y(ylim[1])}"/>`;},\n finish(){ $(id).innerHTML=`<svg role="img" aria-label="${esc(xlabel+\' versus \'+ylabel)}" viewBox="0 0 ${W} ${H}">${body}<g clip-path="url(#${clip})">${content}</g></svg>`;}\n};return api;}\nfunction panel(id,title,caption,wide=false){return `<section class="plot ${wide?\'wide\':\'\'}"><h3>${title}</h3><div id="${id}"></div><div class="caption">${caption}</div></section>`;}\nfunction metric(key,value,foot){return `<div class="metric"><div class="key">${key}</div><div class="value">${value}</div><div class="foot">${foot}</div></div>`;}\nfunction addSlider(key,label,min,max,step,formatter){const div=document.createElement(\'div\');div.className=\'control\';div.innerHTML=`<label for="c-${key}"><span>${label}</span><output id="v-${key}" for="c-${key}"></output></label><input id="c-${key}" name="${key}" type="range" min="${min}" max="${max}" step="${step}" value="${state[key]}">`;$(\'controls\').append(div);const input=$(\'c-\'+key);input.addEventListener(\'input\',()=>{pause();state[key]=key===\'N\'?parseInt(input.value,10):parseFloat(input.value);render();});input._format=formatter;}\nfunction addSelect(key,label,options){const div=document.createElement(\'div\');div.className=\'control\';div.innerHTML=`<label for="c-${key}">${label}</label><select id="c-${key}" name="${key}">${options.map(([v,t])=>`<option value="${v}">${t}</option>`).join(\'\')}</select>`;$(\'controls\').append(div);$(\'c-\'+key).value=state[key];$(\'c-\'+key).addEventListener(\'change\',e=>{pause();state[key]=e.target.value;render();});}\nfunction syncControls(){for(const key of Object.keys(state)){const c=$(\'c-\'+key);if(!c)continue;c.value=state[key];const out=$(\'v-\'+key);if(out)out.textContent=c._format(state[key]);}document.querySelectorAll(\'[data-n]\').forEach(b=>b.classList.toggle(\'active\',+b.dataset.n===state.N));}\nfunction presets(){const div=$(\'presets\');div.className=\'preset\';div.innerHTML=\'<span>Quick N:</span>\'+Ns.map(n=>`<button type="button" data-n="${n}">${n}</button>`).join(\'\');div.querySelectorAll(\'button\').forEach(b=>b.addEventListener(\'click\',()=>{pause();state.N=+b.dataset.n;render();}));}\nfunction lattice(){const N=state.N;const shown=Math.min(N,10);const W=1000,H=112;let txt=\'\';if(MODE===\'spacing\'){const scale=840/(9*10);for(let j=0;j<shown;j++){const xx=70+j*state.a*scale;if(j)txt+=`<line x1="${70+(j-1)*state.a*scale}" y1="43" x2="${xx}" y2="43" stroke="currentColor" opacity=".35"/>`;txt+=`<circle cx="${xx}" cy="43" r="7" fill="currentColor"/><text x="${xx}" y="73" text-anchor="middle" class="tick">${j+1}</text>`;}txt+=`<text x="70" y="100" class="tick">Fixed spatial scale • ${shown<N?\'first \'+shown+\' sites of \'+N:N+\' sites\'} • a = ${fixed(state.a)} Å${shown<N?\' • …\':\'\'}</text>`;}else{for(let j=0;j<shown;j++){const xx=shown===1?500:70+j*860/(shown-1);if(j)txt+=`<line x1="${70+(j-1)*860/(shown-1)}" y1="39" x2="${xx}" y2="39" stroke="currentColor" opacity=".35"/>`;txt+=`<circle cx="${xx}" cy="39" r="7" fill="currentColor"/><text x="${xx}" y="69" text-anchor="middle" class="tick">${shown<N && j===shown-1?\'N\':j+1}</text>`;}if(N>shown)txt+=`<rect x="860" y="27" width="40" height="25" fill="white"/><text x="880" y="44" text-anchor="middle" fill="currentColor">⋯</text>`;txt+=`<text x="500" y="101" text-anchor="middle" class="tick">${N===1?\'One isolated site\':N+\' sites, one orbital per site; open ends\'+(N>shown?\' (chain drawing abbreviated)\':\'\')}</text>`;} $(\'lattice\').innerHTML=`<svg role="img" aria-label="Open chain schematic" viewBox="0 0 ${W} ${H}">${txt}</svg>`;}\nfunction matrix(){const n=Math.min(state.N,7),W=550,H=380,s=Math.min(42,260/n),x0=(W-s*n)/2,y0=38;let txt=\'\';for(let i=0;i<n;i++){for(let j=0;j<n;j++){const val=i===j?state.eps:Math.abs(i-j)===1?-state.mag:0;const xx=x0+j*s,yy=y0+i*s;txt+=`<rect x="${xx}" y="${yy}" width="${s-1}" height="${s-1}" rx="3" class="${i===j?\'matrixdiag\':Math.abs(i-j)===1?\'matrixbond\':\'\'}" ${Math.abs(i-j)>1?\'fill="none"\':\'\'}/><text x="${xx+s/2}" y="${yy+s/2+4}" font-size="${n<4?16:13}" text-anchor="middle" class="${val===0?\'matrixzero\':\'matrixval\'}">${fmt(val,2)}</text>`;}}\nconst yb=y0+n*s;txt+=`<path d="M${x0-5},${y0-4} h-8 v${n*s+5} h8 M${x0+n*s+4},${y0-4} h8 v${n*s+5} h-8" fill="none" stroke="currentColor"/><text x="${W/2}" y="${Math.max(yb+32,340)}" text-anchor="middle" class="tick">${state.N>n?\'Upper-left \'+n+\' × \'+n+\' block of the \'+state.N+\' × \'+state.N+\' matrix\':\'Full \'+state.N+\' × \'+state.N+\' Hamiltonian\'} (eV)</text>`;$(\'matrix\').innerHTML=`<svg role="img" aria-label="Nearest-neighbor Hamiltonian" viewBox="0 0 ${W} ${H}">${txt}</svg>`;}\nfunction renderBand(){const N=state.N,t=-state.mag,eps=state.eps;const es=levels(N,eps,t);const q=grid(0,1,401);const spread=Math.max(2.35*Math.abs(t),.15);const p=plot(\'spectrum\',[0,1],[eps-spread,eps+spread],\'Mode coordinate n / (N + 1)\',\'Energy E (eV)\');p.path(q,q.map(s=>eps+2*t*Math.cos(Math.PI*s)),\'ideal-line\');p.hline(eps);p.dots(es.map((_,i)=>(i+1)/(N+1)),es);p.finish();matrix();lattice();const width=es[N-1]-es[0];$(\'metrics\').innerHTML=metric(\'Available states\',N,\'one spin channel\')+metric(\'Nearest-neighbor t\',fmt(t)+\' eV\',\'negative hopping convention\')+metric(\'Finite width W_N\',fmt(width)+\' eV\',\'E_N − E_1\')+metric(\'Bulk width W∞\',fmt(4*Math.abs(t))+\' eV\',\'4 |t|; limiting full width\');$(\'takeaway\').innerHTML=t===0?\'<b>Isolated-atom limit:</b> all N states have energy ε. They are degenerate, not a single state. Set |t| above zero to split the levels.\':`<b>Read the plot:</b> every point is one state. Increasing N inserts more levels into a bounded interval; it does not make the bandwidth grow without limit. The lowest and highest levels are <b>${fmt(es[0])}</b> and <b>${fmt(es[N-1])} eV</b>.`;$(\'status\').textContent=\'Checks: \'+N+\' levels • Σ E_n = N ε to numerical precision • no end-to-end bond.\';window.__lab_metrics={N,t,epsilon:eps,levels:es,width,bulkWidth:4*Math.abs(t)};}\nfunction renderSpacing(){const N=state.N,a=state.a,t=hopping(a),eps=state.eps;const es=levels(N,eps,t);const aa=grid(3,10,161);const maxT=Math.abs(hopping(3));const spread=Math.max(2.25*maxT,.15);const fan=plot(\'fan\',[3,10],[eps-spread,eps+spread],\'Lattice spacing a (Å)\',\'Energy E (eV)\');const stride=Math.max(1,Math.ceil(N/64));for(let n=1;n<=N;n+=stride)fan.path(aa,aa.map(v=>eps+2*hopping(v)*Math.cos(n*Math.PI/(N+1))),\'fan\');fan.vline(a);fan.hline(eps);fan.dots(es.map(()=>a),es);fan.finish();const s=plot(\'spacing-spectrum\',[0,1],[eps-spread,eps+spread],\'Mode coordinate n / (N + 1)\',\'Energy E (eV)\');const q=grid(0,1,401);s.path(q,q.map(v=>eps+2*t*Math.cos(Math.PI*v)),\'ideal-line\');s.dots(es.map((_,i)=>(i+1)/(N+1)),es);s.hline(eps);s.finish();lattice();$(\'metrics\').innerHTML=metric(\'Spacing a\',fixed(a)+\' Å\',\'reference a₀ = \'+fmt(CFG.a_ref_A)+\' Å\')+metric(\'Hopping t(a)\',fmt(t)+\' eV\',\'includes the a prefactor\')+metric(\'Bulk width W∞\',fmt(4*Math.abs(t))+\' eV\',\'4 |t(a)|\')+metric(\'Available states\',N,\'unchanged as a is varied\');$(\'takeaway\').innerHTML=\'<b>Separate two effects:</b> N controls how many states exist; the coupling controls how widely they are split. Increasing a weakens |t(a)| in this slider range, so the levels collapse toward ε. The energy axes stay fixed during the a sweep.\';$(\'status\').textContent=(N>64?\'The fan shows every \'+stride+\'th branch; all \'+N+\' current levels are marked. \':\'All \'+N+\' levels are shown. \')+\'This is a parameter sweep, not electron motion in time.\';window.__lab_metrics={N,a,t,levels:es,width:es[N-1]-es[0],bulkWidth:4*Math.abs(t)};}\nfunction renderDos(){const N=state.N,mag=state.mag,t=-mag,eps=state.eps,a=state.a,r=state.eta,sigma=r*mag;const us=grid(-2-6*r,2+6*r,721);const eg=us.map(u=>eps+mag*u);const es=levels(N,eps,t);const rs=levels(N,0,-1);const shape=densityShape(us,rs,r);const key=r.toFixed(5);if(!smoothCache || smoothCache.key!==key){const nq=1024;const rq=Array.from({length:nq},(_,i)=>-2*Math.cos((i+.5)*Math.PI/nq));smoothCache={key,shape:densityShape(us,rq,r)};}const factor=state.norm===\'site\'?1/mag:1/(a*mag);const D=shape.map(v=>v*factor);const B=smoothCache.shape.map(v=>v*factor);const area=integral(shape,us);const center=eps+2*mag*state.center,delta=4*mag*state.width,lo=center-delta/2,hi=center+delta/2;const nlo=countAt(lo,es),nhi=countAt(hi,es),count=nhi-nlo;const chosen=es.map(e=>e>lo&&e<=hi);const ylim=[eg[0],eg[eg.length-1]];const p=plot(\'dos-spectrum\',[0,1],ylim,\'Mode coordinate n / (N + 1)\',\'Energy E (eV)\');p.shadeY(lo,hi);const q=grid(0,1,401);p.path(q,q.map(v=>eps+2*t*Math.cos(Math.PI*v)),\'ideal-line\');p.dots(es.map((_,i)=>(i+1)/(N+1)),es,chosen);p.hline(eps-2*mag);p.hline(eps+2*mag);p.finish();let dmax=Math.max(...D,...B)*1.14;const ylabel=state.norm===\'site\'?\'a D(E) [states / (eV · site)]\':\'D(E) [states / (eV · Å)]\';const dp=plot(\'dos\',[0,dmax],ylim,ylabel,\'Energy E (eV)\');dp.shadeY(lo,hi);dp.path(D,eg);const aq=grid(.0008,Math.PI-.0008,801);const ae=aq.map(v=>eps-2*mag*Math.cos(v));const ad=aq.map(v=>factor/(2*Math.PI*Math.sin(v)));dp.path(ad,ae,\'ideal-line\');dp.path(B,eg,\'smooth-line\');dp.hline(eps-2*mag);dp.hline(eps+2*mag);dp.finish();const cp=plot(\'cumulative\',[eg[0],eg[eg.length-1]],[0,1.04],\'Energy E (eV)\',\'Fraction Ω_N(E) / N\',true);cp.shadeX(lo,hi);const sx=[eg[0]],sy=[0];for(let i=0;i<N;i++){sx.push(es[i],es[i]);sy.push(i/N,(i+1)/N);}sx.push(eg[eg.length-1]);sy.push(1);cp.path(sx,sy);cp.path(eg,eg.map(E=>Math.acos(Math.max(-1,Math.min(1,(eps-E)/(2*mag))))/Math.PI),\'ideal-line\');cp.vline(lo);cp.vline(hi);cp.finish();const avg=count/(N*a*delta);const displayAvg=state.norm===\'site\'?a*avg:avg;$(\'metrics\').innerHTML=metric(\'States in the window\',count+\' / \'+N,\'exact count: E_low < E_n ≤ E_high\')+metric(\'Window width ΔE\',fmt(delta)+\' eV\',\'center = \'+fmt(center)+\' eV\')+metric(\'Average DOS in window\',fmt(displayAvg),state.norm===\'site\'?\'ΔΩ / (N ΔE)\':\'ΔΩ / (N a ΔE)\')+metric(\'Area check\',area.toFixed(6),\'a ∫ Dσ(E) dE = 1\');$(\'takeaway\').innerHTML=`<b>Scan the same-width window:</b> from <b>${fmt(lo)}</b> to <b>${fmt(hi)} eV</b>, Ω_N increases from <b>${nlo}</b> to <b>${nhi}</b>. Near a band edge, the E_n curve is flatter, so a small energy interval can contain more states. Changing σ changes the drawn peaks, not these exact counts.`;$(\'status\').textContent=\'L = N a = \'+fmt(N*a)+\' Å • \'+N+\' states total • σ = \'+fmt(sigma)+\' eV • per-length DOS area = 1/a = \'+fmt(1/a)+\' Å⁻¹. Energy axes match; the horizontal DOS scale adjusts for readability.\';window.__lab_metrics={N,a,t,sigma,low:lo,high:hi,count,nlo,nhi,areaPerSite:area,levels:es,D,energyGrid:eg,normalization:state.norm};}\nfunction render(){syncControls();if(MODE===\'band\')renderBand();else if(MODE===\'spacing\')renderSpacing();else renderDos();resizeFrame();}\nfunction step(dir){if(MODE===\'spacing\'){state.a=+(state.a-dir*.1).toFixed(2);if(state.a<3-1e-8)state.a=10;if(state.a>10+1e-8)state.a=3;}else if(MODE===\'dos\'&&state.sweep===\'window\'){state.center=+(state.center+dir*.04).toFixed(3);if(state.center>1.1)state.center=-1.1;if(state.center< -1.1)state.center=1.1;}else{if(dir>0)state.N=Ns.find(n=>n>state.N)||Ns[0];else state.N=[...Ns].reverse().find(n=>n<state.N)||Ns[Ns.length-1];}render();}\nfunction pause(){playing=false;if(raf!==null)cancelAnimationFrame(raf);raf=null;$(\'play\').textContent=\'▶ Play\';}\nfunction animate(timestamp){if(!playing)return;if(timestamp-lastFrame>=650/speed){lastFrame=timestamp;step(direction);}raf=requestAnimationFrame(animate);}\nfunction start(){playing=true;lastFrame=performance.now();$(\'play\').textContent=\'❚❚ Pause\';if(raf!==null)cancelAnimationFrame(raf);raf=requestAnimationFrame(animate);}\n$(\'play\').onclick=()=>playing?pause():start();$(\'reverse\').onclick=()=>{direction*=-1;start();};$(\'back\').onclick=()=>{pause();step(-1);};$(\'forward\').onclick=()=>{pause();step(1);};$(\'reset\').onclick=()=>{pause();direction=1;state={...initial};smoothCache=null;render();};$(\'speed\').onchange=e=>{speed=+e.target.value;};\nfunction resizeFrame(){try{if(window.frameElement)window.frameElement.style.height=(document.body.getBoundingClientRect().height+8)+\'px\';}catch(e){/* Standalone or a sandboxed preview: normal iframe scrolling remains available. */}}\nfunction init(){\nif(MODE===\'band\'){\n $(\'kicker\').textContent=\'Experiment 1\';$(\'title\').textContent=\'Add atoms. Watch an energy band form.\';$(\'intro\').textContent=\'Each site contributes one basis state. Turn on nearest-neighbor coupling and increase N to see those states split into a densely sampled band.\';$(\'formula\').innerHTML=\'H<sub>jj</sub> = ε, &nbsp; H<sub>j,j+1</sub> = H<sub>j+1,j</sub> = t &lt; 0 &nbsp; → &nbsp; E<sub>n</sub> = ε + 2t cos[nπ/(N+1)]\';\n addSlider(\'N\',\'Number of sites N\',1,256,1,x=>String(x));addSlider(\'mag\',\'Coupling magnitude |t|\',0,1.2,.01,x=>fixed(x)+\' eV\');addSlider(\'eps\',\'On-site energy ε\',-1,1,.02,x=>fixed(x)+\' eV\');presets();$(\'sweepnote\').textContent=\'N sweep; the N slider also scrubs frames. Not time evolution.\';\n $(\'plots\').innerHTML=panel(\'spectrum\',\'One point = one energy eigenstate\',\'Points: exact finite-chain levels. Dashed curve: the cosine envelope, not extra states.\')+panel(\'matrix\',\'The Hamiltonian being solved\',\'Only the main diagonal and the two nearest-neighbor diagonals are nonzero.\');$(\'method\').innerHTML=\'<p>The browser evaluates the exact eigenvalues of the real, open tridiagonal matrix. The notebook independently builds the matrix and diagonalizes it with NumPy. For the negative t used here, n = 1 is the lowest-energy state.</p><p>W<sub>N</sub> = 4|t| cos[π/(N+1)], whereas the limiting full bandwidth is W<sub>∞</sub> = 4|t|. At t = 0 all N states are degenerate at ε.</p>\';\n}else if(MODE===\'spacing\'){\n $(\'kicker\').textContent=\'Experiment 2\';$(\'title\').textContent=\'Pull the atoms apart. Collapse the band.\';$(\'intro\').textContent=\'Keep the number of orbitals fixed and vary only the spacing-dependent coupling. The fan records all level energies versus a; the vertical line marks the current spacing.\';$(\'formula\').innerHTML=\'t(a) = t<sub>ref</sub> (a/a<sub>ref</sub>) exp[−κ(a − a<sub>ref</sub>)] &nbsp; &nbsp; W<sub>∞</sub>(a) = 4|t(a)|\';\n addSlider(\'N\',\'Number of sites N\',1,256,1,x=>String(x));addSlider(\'a\',\'Spacing a / frame slider\',3,10,.05,x=>fixed(x)+\' Å\');addSlider(\'kappa\',\'Decay constant κ\',.6,1.8,.05,x=>fixed(x)+\' Å⁻¹\');$(\'sweepnote\').textContent=\'Play decreases a; Reverse increases a. Not time evolution.\';\n $(\'plots\').innerHTML=panel(\'fan\',\'The energy-level fan\',\'The energy range is fixed for the full a sweep at the selected κ.\')+panel(\'spacing-spectrum\',\'Current finite-chain spectrum\',\'N does not change when you move the a slider.\');$(\'method\').innerHTML=\'<p>The approximate coupling is proportional to a exp(−κa). The displayed equation is that expression normalized to an illustrative reference value t<sub>ref</sub> = \'+fmt(CFG.t_ref_eV)+\' eV at a<sub>ref</sub> = \'+fmt(CFG.a_ref_A)+\' Å. It preserves the prefactor a, not just the dominant exponential.</p><p>The reference hopping and κ are model inputs, not a fit to a particular material. The notebook retains only nearest-neighbor coupling throughout.</p>\';\n}else{\n $(\'kicker\').textContent=\'Experiment 3\';$(\'title\').textContent=\'Count states. Build the density of states.\';$(\'intro\').textContent=\'The shaded energy window is linked across all three plots. Drag it from the band center to an edge, or animate N to turn resolved peaks into a smooth DOS.\';$(\'formula\').innerHTML=\'Ω<sub>N</sub>(E) = # {E<sub>n</sub> ≤ E} &nbsp; &nbsp; D<sub>σ</sub>(E) = (1/Na) Σ<sub>n</sub> G<sub>σ</sub>(E − E<sub>n</sub>) &nbsp; &nbsp; D<sub>∞</sub>(E) = 1/[πa √(4t² − (E − ε)²)]\';\n addSlider(\'N\',\'Number of sites N\',1,256,1,x=>String(x));addSlider(\'mag\',\'Coupling magnitude |t|\',.05,1.2,.01,x=>fixed(x)+\' eV\');addSlider(\'a\',\'Spacing a (t held fixed)\',3,10,.05,x=>fixed(x)+\' Å\');addSlider(\'eta\',\'Display resolution σ / |t|\',.02,.30,.01,x=>fixed(x));addSlider(\'center\',\'Window center (E_c − ε)/(2|t|)\',-1.1,1.1,.01,x=>fixed(x));addSlider(\'width\',\'Window width ΔE / (4|t|)\',.02,.35,.01,x=>fixed(x));addSelect(\'norm\',\'DOS normalization\',[[\'length\',\'Per length: D(E)\'],[\'site\',\'Per site: a D(E)\']]);addSelect(\'sweep\',\'Animate which quantity?\',[[\'N\',\'Number of sites N\'],[\'window\',\'Position of the energy window\']]);presets();$(\'sweepnote\').textContent=\'N or energy-window sweep; not time evolution.\';$(\'lattice\').style.display=\'none\';\n const legend=\'<div class="legend"><span><i class="swatch"></i>finite N, Gaussian broadened</span><span><i class="swatch ideal"></i>ideal bulk</span><span><i class="swatch smooth"></i>bulk, same broadening</span></div>\';\n $(\'plots\').innerHTML=panel(\'dos-spectrum\',\'Which levels fall in this energy window?\',\'Highlighted points are counted states, not occupied electrons.\')+\'<section class="plot"><h3>From discrete peaks to the continuum</h3><div id="dos"></div>\'+legend+\'<div class="caption">Ideal bulk diverges at both band edges and is clipped by the plot boundary. Gaussian tails beyond the band are display broadening, not extra eigenstates.</div></section>\'+panel(\'cumulative\',\'Ω_N(E): the staircase whose derivative is the DOS\',\'Each step is exactly one state (height 1/N here). The smooth dashed curve is the large-N cumulative fraction.\',true);\n $(\'method\').innerHTML=\'<p>L = Na and one spin channel are used throughout. Every Gaussian G<sub>σ</sub> has unit area, so ∫D<sub>σ</sub>dE = N/L = 1/a, and the per-site area is one. A finite chain has an exact delta-function DOS; σ is an explicitly introduced display resolution, not temperature or a scattering time.</p><p>At fixed σ, increasing N approaches the <em>broadened</em> bulk curve. The unbroadened singular curve is a distinct limit. The broadened bulk reference is evaluated by midpoint quadrature in the continuum mode coordinate q.</p><p>This a slider changes only the length normalization while holding t fixed. Use Experiment 2 to couple a and t physically through the chosen approximate t(a).</p>\';\n}\nrender();new ResizeObserver(resizeFrame).observe(document.body);window.addEventListener(\'resize\',resizeFrame);\n}\n// Exposed only for transparent numerical checks / browser-console exploration.\nwindow.latticeLab={getState:()=>({...state}),levels,countAt,densityShape,pause,\n setState:updates=>{pause();state={...state,...updates};render();}};\ninit();\n</script>\n</body>\n</html>\n'


LAB_DOCS = {}

def make_browser_document(mode):
    if mode not in ("band", "spacing", "dos"):
        raise ValueError("Unknown experiment: " + str(mode))
    config = dict(SETTINGS)
    # Use Matplotlib's existing default plotting cycle rather than a new chart palette.
    config["plot_colors"] = plt.rcParams["axes.prop_cycle"].by_key()["color"][:3]
    return LAB_TEMPLATE.replace("__CONFIG__", json.dumps(config)).replace("__MODE__", json.dumps(mode))


def iframe_for_document(document, title, height=1000):
    # srcdoc keeps scripts isolated from the notebook DOM. Same-origin permission
    # lets this self-contained, locally generated frame resize itself to its content.
    return (f'<iframe title="{html.escape(title, quote=True)}" '
            f'srcdoc="{html.escape(document, quote=True)}" '
            f'sandbox="allow-scripts allow-same-origin" '
            f'style="width:100%;height:{height}px;border:1px solid #d9e2e6;'
            'border-radius:12px;background:white" loading="eager"></iframe>')


def show_browser_lab(mode):
    document = make_browser_document(mode)
    LAB_DOCS[mode] = document
    # Also save a standalone copy that opens in any browser.
    (OUTPUT_DIR / f"atoms_to_a_band_{mode}.html").write_text(document, encoding="utf-8")
    display(HTML("<div>" + iframe_for_document(document, "From atoms to a band: " + mode,
                                             height=1450 if mode == "dos" else 1050) + "</div>"))

In [ ]:
# Optional GIF rendering helpers. These are parameter sweeps, not time evolution.
# Each GIF contains one chart, with fixed axes throughout the sweep.

def _save_gif(fig, frames, update, filename, duration_ms):
    path = OUTPUT_DIR / filename
    images = []
    for frame in frames:
        update(frame)
        fig.canvas.draw()
        images.append(PILImage.fromarray(np.asarray(fig.canvas.buffer_rgba()).copy()).convert("RGB"))
    durations = [duration_ms] * len(images)
    durations[0] = max(duration_ms, 650)
    durations[-1] = max(duration_ms, 1300)
    images[0].save(path, save_all=True, append_images=images[1:],
                   duration=durations, loop=0, optimize=True)
    plt.close(fig)
    return path


def make_band_gif():
    eps, t = SETTINGS["epsilon_eV"], SETTINGS["t_eV"]
    frames = [1, 2, 3, 4, 6, 8, 12, 16, 24, 32, 48, 64, 96, 128, 192, 256]
    q = np.linspace(0, 1, 600)
    fig = plt.figure(figsize=(7.8, 4.8), dpi=100)
    ax = fig.add_subplot(111)
    ax.plot(q, eps + 2*t*np.cos(np.pi*q), "--", lw=1.6, label="Cosine envelope")
    points, = ax.plot([], [], "o", ms=4, label="Finite-chain eigenvalues")
    spread = max(2.35*abs(t), 0.15)
    ax.set(xlim=(0, 1), ylim=(eps-spread, eps+spread),
           xlabel=r"Mode coordinate $n/(N+1)$", ylabel="Energy E (eV)")
    ax.grid(True, alpha=0.25)
    ax.legend(loc="upper left", fontsize=9)
    title = ax.set_title("Parameter sweep\nOne state per orbital", fontsize=12)
    fig.tight_layout()
    def update(N):
        es = exact_levels(N, eps, t)
        points.set_data(np.arange(1, N+1)/(N+1), es)
        title.set_text(f"Add atoms: N = {N} states\nFinite width = {np.ptp(es):.3f} eV; bulk width = {4*abs(t):.3f} eV")
    return _save_gif(fig, frames, update, "01_band_formation.gif", 430)


def make_spacing_gif():
    N, eps = SETTINGS["N"], SETTINGS["epsilon_eV"]
    kwargs = {k: SETTINGS[k] for k in ("t_ref_eV", "a_ref_A", "kappa_invA")}
    frames = np.r_[np.linspace(10, 3, 31), np.linspace(3, 10, 31)[1:]]
    max_t = abs(float(hopping_from_spacing(3.0, **kwargs)))
    q = np.linspace(0, 1, 600)
    fig = plt.figure(figsize=(7.8, 4.8), dpi=100)
    ax = fig.add_subplot(111)
    curve, = ax.plot([], [], "--", lw=1.5, label="Current cosine envelope")
    points, = ax.plot([], [], "o", ms=4, label=f"N = {N} states (fixed)")
    spread = max(2.25*max_t, 0.15)
    ax.set(xlim=(0, 1), ylim=(eps-spread, eps+spread),
           xlabel=r"Mode coordinate $n/(N+1)$", ylabel="Energy E (eV)")
    ax.grid(True, alpha=0.25)
    ax.legend(loc="upper left", fontsize=9)
    title = ax.set_title("Parameter sweep\nOne state per orbital", fontsize=12)
    fig.tight_layout()
    def update(a):
        t = float(hopping_from_spacing(a, **kwargs))
        curve.set_data(q, eps + 2*t*np.cos(np.pi*q))
        points.set_data(np.arange(1, N+1)/(N+1), exact_levels(N, eps, t))
        title.set_text(f"Vary spacing: a = {a:.2f} Å; t(a) = {t:.4f} eV\nN stays fixed; only the level splitting changes")
    return _save_gif(fig, frames, update, "02_spacing_and_bandwidth.gif", 140)


def make_dos_gif():
    eps, t, a = (SETTINGS[k] for k in ("epsilon_eV", "t_eV", "a_A"))
    sigma = SETTINGS["eta_over_abs_t"]*abs(t)
    frames = [2, 3, 4, 8, 16, 32, 64, 128, 256]
    E = np.linspace(eps-2*abs(t)-6*sigma, eps+2*abs(t)+6*sigma, 1801)
    all_dos = {N: gaussian_dos(E, exact_levels(N, eps, t), a, sigma) for N in frames}
    db = smoothed_bulk_dos(E, eps, t, a, sigma)
    q = np.linspace(0.0008, np.pi-0.0008, 1601)
    ideal_E = eps-2*abs(t)*np.cos(q)
    ideal_D = 1/(2*np.pi*a*abs(t)*np.sin(q))
    fig = plt.figure(figsize=(7.8, 5.0), dpi=100)
    ax = fig.add_subplot(111)
    line, = ax.plot([], [], lw=2, label="Finite N, Gaussian broadened")
    ax.plot(ideal_D, ideal_E, "--", lw=1.4, label="Ideal bulk (edge divergence clipped)")
    ax.plot(db, E, ":", lw=2, label="Bulk, same broadening")
    xmax = max(float(np.max(d)) for d in all_dos.values())*1.12
    ax.set(xlim=(0, xmax), ylim=(E[0], E[-1]),
           xlabel=r"DOS $D(E)$ [states / (eV Å)]", ylabel="Energy E (eV)")
    ax.grid(True, alpha=0.25)
    ax.legend(loc="upper right", fontsize=8)
    title = ax.set_title("Parameter sweep\nOne state per orbital", fontsize=12)
    fig.tight_layout()
    def update(N):
        line.set_data(all_dos[N], E)
        title.set_text(f"DOS from N = {N} levels; σ = {sigma:.3f} eV\nFixed resolution and axes; ∫ D(E) dE = 1/a")
    return _save_gif(fig, frames, update, "03_density_of_states.gif", 650)


GIF_PATHS = {}

def show_gif(kind):
    if not MAKE_GIFS:
        print("GIF rendering is disabled. The browser experiments remain available.")
        return
    makers = {"band": make_band_gif, "spacing": make_spacing_gif, "dos": make_dos_gif}
    path = makers[kind]()
    GIF_PATHS[kind] = path
    display(Image(data=path.read_bytes(), format="gif"))
    print(f"Saved: {path.name}")

## 2. Experiment 1 — Add atoms and form a band

**Predict first:** With $t=-0.4$ eV and $\varepsilon=0$, what are the two energies for $N=2$? If the atom count doubles repeatedly, will the full bandwidth also double repeatedly?

Use **Play**, **Reverse**, or the $N$ slider. Then set $|t|=0$ and increase it again. Finally change $\varepsilon$ without changing $t$.

Every dot represents one state. The continuum band edges and the finite-chain width follow directly from the exact spectrum:

$$E_{\rm low}=\varepsilon-2|t|,\qquad E_{\rm high}=\varepsilon+2|t|,$$
$$W_\infty=4|t|,\qquad W_N=4|t|\cos\left(\frac{\pi}{N+1}\right).$$

Both widths follow directly from the exact spectrum: the outermost levels are $n=1$ and $n=N$.

In [ ]:
show_browser_lab("band")

### GIF fallback — Band formation
The axes remain fixed as $N$ changes. This loops through a precomputed parameter sweep and does not require JavaScript.

In [ ]:
show_gif("band")

## 3. Experiment 2 — Change the atomic spacing

Nearest-neighbor coupling comes from the overlap of the decaying atomic orbitals. An approximate hopping integral contains the dependence

$$t(a)\approx-\frac{\hbar^2\kappa^3}{2m}\,a\,e^{-\kappa a}.$$

To explore that dependence without claiming an absolute microscopic calibration, we normalize to a reference value:

$$\boxed{t(a)=t_{\rm ref}\frac{a}{a_{\rm ref}}
\exp[-\kappa(a-a_{\rm ref})]}.$$

This preserves the **$a$ prefactor**, not just the dominant exponential. We keep $\varepsilon$, the number of states, and the nearest-neighbor approximation fixed.

**Predict first:** As atoms are pulled farther apart, do states disappear, or do their energies move closer together? Try the spacing slider, then vary $\kappa$. The selected slider range keeps $a>1/\kappa$, where the chosen $|t(a)|$ decreases with increasing $a$.

In [ ]:
show_browser_lab("spacing")

### GIF fallback — Spacing and bandwidth
Here $N$ is fixed at the value in `SETTINGS`. Both axes remain fixed while the atoms are brought closer and then pulled apart.

In [ ]:
show_gif("spacing")

## 4. Density of states: start by counting

For the finite chain, define the exact staircase count

$$\Omega_N(E)=\sum_{n=1}^{N}\Theta(E-E_n).$$

We count a level **at or below** $E$; this only fixes the convention at each step. The density of states per unit length is then

$$D_N(E)=\frac{1}{L}\frac{d\Omega_N}{dE}
=\frac{1}{Na}\sum_{n=1}^{N}\delta(E-E_n).$$

A finite chain therefore has **discrete delta peaks**. To draw them, this notebook introduces a normalized Gaussian of standard deviation $\sigma$:

$$G_\sigma(\Delta E)=\frac{\exp[-\Delta E^2/(2\sigma^2)]}{\sqrt{2\pi}\sigma},
\qquad D_{N,\sigma}(E)=\frac{1}{Na}\sum_nG_\sigma(E-E_n).$$

This broadening is a **plotting resolution**, not temperature, a lifetime, or extra states outside the band. The exact cumulative count does not depend on $\sigma$.

### The large-$N$ curve

With the same state-count normalization, the continuum fraction is

$$F_\infty(E)=\frac{1}{\pi}\cos^{-1}\left(\frac{\varepsilon-E}{2|t|}\right)$$

inside the band, and it is $0$ below and $1$ above the band. Differentiating gives

$$\boxed{D_\infty(E)=\frac{1}{\pi a\sqrt{4t^2-(E-\varepsilon)^2}}}$$

for $|E-\varepsilon|<2|t|$, with zero DOS outside. The band-edge divergences are integrable. The notebook uses a nonnegative state count and distinguishes the exact finite staircase from the continuum interpolation.

**Two normalization checks:**

$$\int D_N(E)\,dE=\frac{N}{L}=\frac{1}{a},\qquad
\int aD_N(E)\,dE=1.$$

The first is the DOS **per length**. The second is **per site**. No factor of two for spin has been inserted.

## 5. Experiment 3 — A moving energy window

**Predict first:** For an energy window of fixed width $\Delta E$, will there be more states near the band center or near a band edge?

The window is linked across the spectrum, DOS, and cumulative-count plots. Its exact count is

$$\Delta\Omega=\Omega_N(E_{\rm high})-\Omega_N(E_{\rm low}),\qquad
\overline D=\frac{\Delta\Omega}{L\Delta E}.$$

Compare small and large $N$, then adjust $\sigma$ while holding $N$ fixed. The ideal bulk DOS is shown alongside a bulk curve with **the same Gaussian broadening**, so the comparison does not confuse finite size with display resolution.

In this experiment, changing $a$ holds $t$ fixed to isolate **normalization**. Experiment 2 is the place to explore the coupled $a\mapsto t(a)$ dependence.

In [ ]:
show_browser_lab("dos")

### GIF fallback — Discrete peaks become a smooth DOS
The broadening and both axes remain fixed as $N$ increases. At fixed broadening, the finite-chain curve approaches the **broadened bulk** reference, not an infinitely sharp singular curve.

In [ ]:
show_gif("dos")